# 01_hello_world — Minimal Action

The simplest Action: a single `@summary_aspect`, stubs instead of params and result.

**Goal:** show a working structure with a minimum of new concepts. Every line here is required — nothing extra, nothing optional.

| Concept | Description |
|---------|-------------|
| `BaseDomain` | Logical grouping of operations in the system |
| `@meta` | Required class decorator: description and domain |
| `@check_roles` | Required access declaration (`GuestRole` = open to everyone) |
| `BaseAction[P, R]` | Base class for all operations |
| `ParamsStub` | Input data stub (no fields) |
| `ResultStub` | Result stub (no fields) |
| `@summary_aspect` | The single exit point of an Action, returns Result |
| `ActionProductMachine` | The machine that runs the pipeline |
| `Context()` | Call context (user, metadata, etc.; empty here) |

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, ParamsStub, ResultStub
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Step 1 — Domain

`BaseDomain` is a logical group for operations of one subject area. Every Action must belong to a domain via `@meta(domain=...)`.

> **Naming rule:** the class name must end with `Domain`. Try renaming to `Greeting` — you'll get `NamingSuffixError` immediately, before the application even starts.

In [ ]:
class GreetingDomain(BaseDomain):
    name = "greeting"
    description = "Greetings domain"

## Step 2 — Action

Two required decorators — the machine refuses to run without both:
- `@meta(description=..., domain=...)` — what this operation is and where it lives. The `description` goes into OpenAPI, MCP tools, and the Maxitor graph — not a comment, a system contract.
- `@check_roles(GuestRole)` — who can call it. `GuestRole` means open to everyone, but must be written explicitly — silent absence doesn't count.

> **Naming rule:** the class name must end with `Action`. Try writing `SayHello` — you'll get `NamingSuffixError` at class declaration.

**`@summary_aspect`** — the single exit point. Every Action must have exactly one. Method name must end with `_summary`. Description cannot be empty — `@summary_aspect("")` raises `ValueError`.

In [ ]:
@meta(description="Say hello to the world", domain=GreetingDomain)
@check_roles(GuestRole)
class SayHelloAction(BaseAction[ParamsStub, ResultStub]):

    @summary_aspect("Print greeting and return stub")
    async def output_summary(self, params, state, box, connections):
        print("Hello, world!")
        return ResultStub()

## Step 3 — Run

- **`ActionProductMachine`** — the central execution machine. Created once, reused for all calls.
- **`Context()`** — the call context. Currently empty; later it will hold the current user, their roles, and the request `trace_id`. Cannot be `None`.
- **`ParamsStub()`** — an instance of the stub we declared as the input type.

> In Colab, `await` works directly in cells — no need for `asyncio.run()`.

In [ ]:
async def main() -> None:
    machine = ActionProductMachine()
    await machine.run(Context(), SayHelloAction(), ParamsStub())

await main()